# Session 4: RAG End to End

**RAG = Retrieval-Augmented Generation.** Retrieve the relevant pieces of *your* documents, then let the LLM generate an answer *from those pieces*.

Last session we built the retrieval half on clean FAQ rows. Real documents are messier: PDFs with headers, footers, tables, multiple pages, and sometimes **old versions that contradict new ones**. Today we handle all of that.

| Part | What we do |
|---|---|
| 0 | Setup |
| 1 | Why RAG: the LLM alone does not know our documents |
| 2 | Load PDFs and look at what was really extracted |
| 3 | Clean and attach useful metadata |
| 4 | Split into chunks (and check the embedding token limit) |
| 5 | Embed and store in Chroma |
| 6 | Retrieve: similarity, MMR, score threshold, filters |
| 7 | Generate grounded answers with citations |
| 8 | Failure modes, and how to debug them |
| 9 | Evaluate with a small test set |
| 10 | Capstone part 1: RAG over your own documents |
| 11 | Exercises |

**Documents** (`docs/`): four policy PDFs of the fictional Sahyadri Digital Bank, including a **superseded 2024 loan policy** that contradicts the 2026 one.

## Setup: run this cell first

Choose the provider and model in the cell below. The API key is **not** written here: it stays in the `.env` file in this folder, for example:

```
OPENROUTER_API_KEY=sk-or-v1-...
```

In [1]:
# ===================== SETUP: run this cell first =====================
PROVIDER = "openrouter"    # "openrouter", "gemini" or "openai"
MODEL = ""                 # leave empty to use the default model below

import os
from dotenv import load_dotenv

# read the .env file in this folder (override=True picks up edits without restarting the kernel)
if not os.path.exists(".env"):
    raise FileNotFoundError("No .env file in this folder. Create it and add your API key.")
load_dotenv(".env", override=True)

if PROVIDER == "openrouter":
    KEY_NAME = "OPENROUTER_API_KEY"
    MODEL = MODEL or "google/gemini-2.5-flash-lite"
    BASE_URL = "https://openrouter.ai/api/v1"
elif PROVIDER == "gemini":
    KEY_NAME = "GOOGLE_API_KEY"
    MODEL = MODEL or "gemini-2.5-flash-lite"
    BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
elif PROVIDER == "openai":
    KEY_NAME = "OPENAI_API_KEY"
    MODEL = MODEL or "gpt-4.1-mini"
    BASE_URL = "https://api.openai.com/v1"
else:
    raise ValueError('PROVIDER must be "openrouter", "gemini" or "openai"')

API_KEY = os.getenv(KEY_NAME)
if not API_KEY:
    raise ValueError(f"{KEY_NAME} is missing. Add it to the .env file in this folder.")


def get_chat_model(temperature=0, **kwargs):
    """LangChain chat model for the chosen provider."""
    if PROVIDER == "openrouter":
        from langchain_openrouter import ChatOpenRouter
        return ChatOpenRouter(model=MODEL, api_key=API_KEY, base_url=BASE_URL, temperature=temperature, **kwargs)
    elif PROVIDER == "gemini":
        from langchain_google_genai import ChatGoogleGenerativeAI
        return ChatGoogleGenerativeAI(model=MODEL, api_key=API_KEY, temperature=temperature, **kwargs)
    else:
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=MODEL, api_key=API_KEY, base_url=BASE_URL, temperature=temperature, **kwargs)


def get_openai_client():
    """Plain OpenAI SDK client pointed at the chosen provider. Returns (client, model)."""
    from openai import OpenAI
    return OpenAI(api_key=API_KEY, base_url=BASE_URL), MODEL


def show_config():
    """Print the settings without showing the full key."""
    print(f"provider : {PROVIDER}")
    print(f"model    : {MODEL}")
    print(f"key      : {KEY_NAME} = {API_KEY[:6]}...{API_KEY[-2:]} ({len(API_KEY)} chars)")
    print(f"endpoint : {BASE_URL}")


show_config()

provider : openrouter
model    : google/gemini-2.5-flash-lite
key      : OPENROUTER_API_KEY = sk-or-...1d (73 chars)
endpoint : https://openrouter.ai/api/v1


## Part 0: Setup

In [3]:
import os
import re
import glob

import numpy as np
import pandas as pd

from importlib.metadata import version

for pkg in ["langchain-core", "langchain-text-splitters", "langchain-chroma", "pypdf", "chromadb"]:
    print(f"{pkg:25s} {version(pkg)}")

llm = get_chat_model(temperature=0)

langchain-core            1.6.7
langchain-text-splitters  1.1.3
langchain-chroma          1.1.0
pypdf                     6.19.0
chromadb                  1.5.9


## Part 1: Why RAG

Ask the LLM directly about our bank's policy:

In [4]:
question = "What is the foreclosure charge on a personal loan at Sahyadri Digital Bank?"
print(llm.invoke(question).content)

I cannot provide you with the specific foreclosure charge on a personal loan at Sahyadri Digital Bank. Here's why:

*   **I am an AI and do not have access to real-time, specific banking policies.** Foreclosure charges are determined by the bank's internal policies, which can change.
*   **This information is usually not publicly disclosed in a general way.** Banks typically provide these details in the loan agreement or upon request from the borrower.

**To find out the foreclosure charge, you will need to contact Sahyadri Digital Bank directly.** Here are the best ways to do that:

1.  **Check Your Loan Agreement:** The most definitive place to find this information is in the loan documents you signed when you took out the personal loan.
2.  **Contact Sahyadri Digital Bank Customer Service:**
    *   **Phone:** Look for their customer service hotline number on their official website or your loan statements.
    *   **Email:** They may have a customer support email address.
    *   **

The model has never seen this bank's documents. It either says it does not know, or worse, gives a **plausible-sounding guess** based on typical Indian bank charges. Both are useless to a customer.

Options to fix this:
- **Fine-tune** on the documents: expensive, slow to update, and does not reliably store facts (Session 1).
- **Paste all documents** into the prompt: works for 2 pages, not for 2,000; costly per question.
- **RAG**: find the 3 to 5 most relevant chunks and send only those. Cheap, easy to update (just re-index a document), and answers can **cite their source**.

```
INDEXING (once per document)          ANSWERING (every question)
load -> clean -> split -> embed       question -> retrieve chunks -> prompt -> LLM -> answer + sources
     -> store in vector DB
```

## Part 2: Load PDFs

A **document loader** just reads a file and returns `Document` objects (text + metadata). For PDFs we write our own with `pypdf`: one `Document` per page. It is only a few lines, and nothing is hidden.

(Older tutorials use `PyPDFLoader` from `langchain_community`. That package is being sunset by LangChain, so we avoid depending on it. Internally it does the same as the function below.)

Always look at what was actually extracted before doing anything else - most RAG problems start here.

In [5]:
from pypdf import PdfReader
from langchain_core.documents import Document

def load_pdf(path):
    reader = PdfReader(path)
    docs = []
    for index, page in enumerate(reader.pages):
        docs.append(Document(
            page_content=page.extract_text() or "",
            metadata={"source": path, "page": index, "total_pages": len(reader.pages)},
        ))
    return docs

pdf_files = sorted(glob.glob("docs/*.pdf"))
print(pdf_files)

pages = []
for path in pdf_files:
    loaded = load_pdf(path)
    print(f"{os.path.basename(path):35s} {len(loaded)} page(s)")
    pages.extend(loaded)

print("\ntotal pages:", len(pages))

['docs/Credit_Card_Terms_2026.pdf', 'docs/Grievance_Redressal_Policy.pdf', 'docs/Personal_Loan_Policy_2024.pdf', 'docs/Personal_Loan_Policy_2026.pdf']
Credit_Card_Terms_2026.pdf          2 page(s)
Grievance_Redressal_Policy.pdf      1 page(s)
Personal_Loan_Policy_2024.pdf       1 page(s)
Personal_Loan_Policy_2026.pdf       2 page(s)

total pages: 6


Metadata of one page. `page` is the **0-based** index from Python's `enumerate` (the first page is `0`). Humans count from 1, so we convert it before citing pages.

In [6]:
print(pages[1].metadata)

{'source': 'docs/Credit_Card_Terms_2026.pdf', 'page': 1, 'total_pages': 2}


Now the text. Look for three kinds of noise:
1. **Header and footer** lines repeated on every page.
2. **Tables** flattened: each cell on its own line, so a row's label and value are separated.
3. **Line breaks** in the middle of sentences.

In [7]:
page = next(p for p in pages if "Fees and charges" in p.page_content)
print(page.page_content[:1100])

Sahyadri Digital Bank | Personal Loan Policy 2024 | Version 2.1 | SUPERSEDED
Page 1
Sahyadri Digital Bank Ltd. | For customer information
Personal Loan Policy 2024 (superseded)
Note: this policy was replaced by the Personal Loan Policy 2026 for all loans sanctioned on or
after 1 April 2026.
1. Scope
This policy applies to personal loans sanctioned between 1 April 2024 and 31 March 2026.
2. Interest rates
Interest rates start at 11.5% per year for applicants with a credit score of 780 and above, and go up to
16% per year. Rates are fixed for the full tenure.
3. Fees and charges
 Charge
Amount
Processing fee
2% of the loan amount, minimum Rs 2,500
Part-prepayment
2% of the amount prepaid
Foreclosure (full prepayment)
4% of the outstanding principal; not allowed before 12 EMIs
Late payment fee
Rs 600 per missed EMI
4. Tenure
Tenure ranges from 12 to 48 months. The loan amount ranges from Rs 50,000 to Rs 15 lakh.



The table survived as alternating lines (`Foreclosure (full prepayment)` then `2% of the outstanding principal...`). That is readable for an LLM as long as the label and value stay in the **same chunk**. Keep this in mind for Part 4.

Quick size check per page:

In [8]:
for p in pages:
    print(f"{os.path.basename(p.metadata['source']):35s} page {p.metadata['page'] + 1}: {len(p.page_content):5d} characters")

Credit_Card_Terms_2026.pdf          page 1:  1589 characters
Credit_Card_Terms_2026.pdf          page 2:  1288 characters
Grievance_Redressal_Policy.pdf      page 1:  1640 characters
Personal_Loan_Policy_2024.pdf       page 1:   923 characters
Personal_Loan_Policy_2026.pdf       page 1:  2193 characters
Personal_Loan_Policy_2026.pdf       page 2:  2250 characters


## Part 3: Clean and add metadata

### Remove repeated header and footer lines

They appear on every page, so they end up in every chunk: they add noise to every embedding and waste tokens. A simple rule-based cleaner is enough here. (For other documents, look at the raw text first and write rules that match them.)

In [9]:
NOISE_PATTERNS = [
    r"^Sahyadri Digital Bank \|.*$",        # running header
    r"^Sahyadri Digital Bank Ltd\. \|.*$",  # footer
    r"^Page \d+$",                           # page number
]

def clean_text(text):
    lines = text.splitlines()
    kept = [line for line in lines if not any(re.match(p, line.strip()) for p in NOISE_PATTERNS)]
    return "\n".join(kept).strip()

before = pages[1].page_content
after = clean_text(before)
print("BEFORE:", repr(before[:160]))
print("AFTER :", repr(after[:160]))
print(f"\n{len(before)} -> {len(after)} characters")

BEFORE: 'Sahyadri Digital Bank | Credit Card Most Important Terms and Conditions\nPage 2\nSahyadri Digital Bank Ltd. | For customer information\n5. Lost or stolen card\nRepo'
AFTER : '5. Lost or stolen card\nReport a lost or stolen card immediately through the app or by calling 1800-000-1111. The card is\nblocked at once. The cardholder has zer'

1288 -> 1154 characters


### Keep only useful metadata

Metadata is what lets us **filter** (only current policies), **cite** (file and page) and **debug** (where did this chunk come from).

In a company this information usually comes from a document register. Here we write it by hand. Note `status`: the 2024 loan policy is **superseded**.

Chroma accepts only simple metadata values (text, numbers, booleans) - no lists or `None`.

In [10]:
DOC_REGISTER = {
    "Personal_Loan_Policy_2026.pdf": {"doc_type": "loan_policy", "year": 2026, "status": "current"},
    "Personal_Loan_Policy_2024.pdf": {"doc_type": "loan_policy", "year": 2024, "status": "superseded"},
    "Credit_Card_Terms_2026.pdf":    {"doc_type": "card_terms", "year": 2026, "status": "current"},
    "Grievance_Redressal_Policy.pdf": {"doc_type": "grievance", "year": 2026, "status": "current"},
}

clean_pages = []
for p in pages:
    file_name = os.path.basename(p.metadata["source"])
    meta = {"source": file_name, "page": int(p.metadata["page"]) + 1}   # human page number
    meta.update(DOC_REGISTER[file_name])
    clean_pages.append(Document(page_content=clean_text(p.page_content), metadata=meta))

print(len(clean_pages), "clean pages")
print(clean_pages[0].metadata)

6 clean pages
{'source': 'Credit_Card_Terms_2026.pdf', 'page': 1, 'doc_type': 'card_terms', 'year': 2026, 'status': 'current'}


## Part 4: Split into chunks

Why not embed whole pages?

1. **The embedding model has a limit.** `all-MiniLM-L6-v2` reads at most **256 tokens**; anything longer is silently cut off and never searchable.
2. **Precision.** One vector for a whole page mixes many topics; a question about foreclosure has to compete with eligibility and documents on the same page.
3. **Cost.** We send only the relevant chunks to the LLM, not whole pages.

Too small is also bad: a chunk with a table label but not its value is useless.

### RecursiveCharacterTextSplitter

It tries to split on paragraph breaks first (`\n\n`), then line breaks (`\n`), then spaces, then characters - so it cuts at the most natural place that keeps chunks under `chunk_size` characters. `chunk_overlap` repeats the end of one chunk at the start of the next, so a fact on a boundary appears whole in at least one chunk.

See it on a small text with tiny sizes:

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

sample = ("Foreclosure is not allowed in the first six months of the loan. "
          "After six EMIs the customer may close the loan at any time by paying the outstanding principal, "
          "interest up to the date of closure, and the foreclosure charge.")

demo_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=30)
for i, chunk in enumerate(demo_splitter.split_text(sample)):
    print(f"chunk {i} ({len(chunk)} chars): {chunk}")

chunk 0 (95 chars): Foreclosure is not allowed in the first six months of the loan. After six EMIs the customer may
chunk 1 (98 chars): six EMIs the customer may close the loan at any time by paying the outstanding principal, interest
chunk 2 (74 chars): principal, interest up to the date of closure, and the foreclosure charge.


Notice the repeated words at the start of each chunk: that is the overlap.

### How big is too big? Count tokens, not characters

The embedding model's own tokenizer tells us the real length. We test three chunk sizes on our pages.

You may see a warning like *"Token indices sequence length is longer than the specified maximum sequence length (321 > 256)"*. That warning **is** the problem we are measuring: those chunks would be cut off when embedded.

In [12]:
from sentence_transformers import SentenceTransformer

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embed_tokenizer = SentenceTransformer(EMBED_MODEL).tokenizer
EMBED_MAX_TOKENS = 256

def n_tokens(text):
    return len(embed_tokenizer(text)["input_ids"])

rows = []
for size in [300, 800, 1500]:
    splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=size // 7)
    chunks = splitter.split_documents(clean_pages)
    tokens = [n_tokens(c.page_content) for c in chunks]
    rows.append({
        "chunk_size (chars)": size,
        "chunks": len(chunks),
        "avg tokens": int(np.mean(tokens)),
        "max tokens": max(tokens),
        "chunks over 256 tokens": sum(t > EMBED_MAX_TOKENS for t in tokens),
    })

pd.DataFrame(rows)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (343 > 256). Running this sequence through the model will result in indexing errors


,chunk_size (chars),chunks,avg tokens,max tokens,chunks over 256 tokens
0,300,39,55,80,0
1,800,15,145,188,0
2,1500,9,237,343,4


At 1500 characters, chunks exceed 256 tokens: the end of each such chunk is **invisible to search**. At 300, facts get scattered across many tiny chunks. **800 characters with 120 overlap** keeps every chunk under the limit while holding a full table section or paragraph together.

There is no universal best size. It depends on the embedding model's limit and on your documents; Part 9 shows how to measure.

In [13]:
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=120, add_start_index=True)
chunks = splitter.split_documents(clean_pages)

print(len(chunks), "chunks")
print(chunks[5].metadata)
print()
print(chunks[5].page_content)

15 chunks
{'source': 'Grievance_Redressal_Policy.pdf', 'page': 1, 'doc_type': 'grievance', 'year': 2026, 'status': 'current', 'start_index': 630}

Banking Ombudsman (RBI)
If no reply within 30 days or the customer is not satisfied with the final response
Customers may approach the RBI Ombudsman through the Complaint Management System website
within one year of receiving the bank's final response, or one year and 30 days from the date of the
complaint if no reply was received.
3. Fraud and unauthorised transactions
Complaints about unauthorised electronic transactions are treated as priority. If reported within 3
working days, the customer has zero liability and the amount is credited as a shadow reversal within
10 working days while the bank investigates. The investigation must be completed within 90 days.
4. Compensation
If a failed ATM or UPI transaction is not reversed within 5 days, the bank pays compensation of Rs


In [14]:
chunks[0]

Document(metadata={'source': 'Credit_Card_Terms_2026.pdf', 'page': 1, 'doc_type': 'card_terms', 'year': 2026, 'status': 'current', 'start_index': 0}, page_content='Credit Card Terms and Conditions (2026)\n1. Card variants and annual fees\n Card\nJoining and annual fee\nSahyadri Classic\nNil\nSahyadri Gold\nRs 999 per year, waived on annual spend above Rs 1.5 lakh\nSahyadri Platinum\nRs 2,999 per year, waived on annual spend above Rs 4 lakh\n2. Billing cycle and payment\nThe statement is generated on the 5th of every month and covers transactions from the 5th of the\nprevious month. Payment is due on the 25th of the same month. The minimum amount due is 5% of\nthe total outstanding or Rs 200, whichever is higher, plus any EMIs and overlimit amount.\nIf the total amount due is paid in full by the due date, no interest is charged on purchases. This gives\nan interest-free period of 20 to 50 days depending on the purchase date.\n3. Interest and finance charges')

Every chunk **inherited** its page's metadata (source, page, status) and gained `start_index` (character position on the page). That is how a chunk knows where it came from.

## Part 5: Embed and store

Same tools as Session 3. Two production habits:

- **Deterministic ids** built from source, page and position. Re-indexing the same document then *updates* chunks instead of duplicating them.
- **Reset by deleting the collection**, not the folder (safe on Windows).

In [15]:
import chromadb
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

DB_DIR = "chroma_db"
COLLECTION = "bank_policies"

client = chromadb.PersistentClient(path=DB_DIR)
if COLLECTION in [c.name for c in client.list_collections()]:
    client.delete_collection(COLLECTION)

embeddings = HuggingFaceEmbeddings(model=EMBED_MODEL, encode_kwargs={"normalize_embeddings": True})

vectorstore = Chroma(
    client=client,
    collection_name=COLLECTION,
    embedding_function=embeddings,
    collection_configuration={"hnsw": {"space": "cosine"}},
)

ids = [f"{c.metadata['source']}-p{c.metadata['page']}-{c.metadata['start_index']}" for c in chunks]
vectorstore.add_documents(chunks, ids=ids)

print("stored chunks:", vectorstore._collection.count())
print("example id   :", ids[0])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

stored chunks: 15
example id   : Credit_Card_Terms_2026.pdf-p1-0


## Part 6: Retrieve

**Rule one of RAG debugging: look at what was retrieved before blaming the LLM.** If the right chunk is not retrieved, no prompt can save the answer.

A helper that prints retrieved chunks with source, page and score:

In [17]:
def show(results, width=110):
    for item in results:
        doc, score = item if isinstance(item, tuple) else (item, None)
        m = doc.metadata
        head = f"[{m['source']} p{m['page']} {m['status']}]"
        score_txt = f" relevance={score:.3f}" if score is not None else ""
        text = " ".join(doc.page_content.split())
        print(f"{head}{score_txt}\n    {text[:width]}...")

show(vectorstore.similarity_search_with_relevance_scores("Can I close my personal loan early and what does it cost?", k=4), width = 200)

[Personal_Loan_Policy_2024.pdf p1 superseded] relevance=0.553
    Personal Loan Policy 2024 (superseded) Note: this policy was replaced by the Personal Loan Policy 2026 for all loans sanctioned on or after 1 April 2026. 1. Scope This policy applies to personal loans...
[Personal_Loan_Policy_2026.pdf p1 current] relevance=0.454
    Personal Loan Policy (effective 1 April 2026) 1. Purpose and scope This policy describes how Sahyadri Digital Bank offers unsecured personal loans to individual customers. It applies to all personal l...
[Personal_Loan_Policy_2026.pdf p2 current] relevance=0.446
    5. Fees and charges All fees below are exclusive of GST at 18%. Charge Amount Processing fee 1.5% of the loan amount, minimum Rs 2,000 Part-prepayment Free after 6 EMIs have been paid Foreclosure (ful...
[Personal_Loan_Policy_2026.pdf p1 current] relevance=0.347
    3. Loan amount and tenure The loan amount ranges from Rs 50,000 to Rs 25 lakh. The maximum amount for an applicant is limited so that

Look at the sources: typically both the **2026** and the **superseded 2024** policy come back, with different foreclosure rules (2% vs 4%). Retrieval found what we asked for; it has no idea which document is valid. We handle this in Part 8.

### Retriever options

| search_type | Behaviour | Use when |
|---|---|---|
| `similarity` | top k most similar | default |
| `mmr` | Maximal Marginal Relevance: relevant **and** different from each other | top results are near-duplicates (overlapping chunks, repeated text) |
| `similarity_score_threshold` | only results above a minimum relevance | you prefer "nothing found" over weak matches |

MMR first fetches `fetch_k` candidates, then picks `k` of them, balancing relevance against similarity to already-picked chunks (`lambda_mult`: 1 = pure relevance, 0 = maximum diversity).

In [21]:
query = "fees and charges"

sim_retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
mmr_retriever = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 4, "fetch_k": 20, "lambda_mult": 1})

print("SIMILARITY:")
show(sim_retriever.invoke(query), width=80)
print("\nMMR:")
show(mmr_retriever.invoke(query), width=80)

SIMILARITY:
[Credit_Card_Terms_2026.pdf p1 current]
    an interest-free period of 20 to 50 days depending on the purchase date. 3. Inte...
[Personal_Loan_Policy_2026.pdf p2 current]
    the customer chooses it, the premium is shown separately and can be paid upfront...
[Credit_Card_Terms_2026.pdf p1 current]
    Credit Card Terms and Conditions (2026) 1. Card variants and annual fees Card Jo...
[Personal_Loan_Policy_2026.pdf p2 current]
    5. Fees and charges All fees below are exclusive of GST at 18%. Charge Amount Pr...

MMR:
[Credit_Card_Terms_2026.pdf p1 current]
    an interest-free period of 20 to 50 days depending on the purchase date. 3. Inte...
[Personal_Loan_Policy_2026.pdf p2 current]
    the customer chooses it, the premium is shown separately and can be paid upfront...
[Credit_Card_Terms_2026.pdf p1 current]
    Credit Card Terms and Conditions (2026) 1. Card variants and annual fees Card Jo...
[Personal_Loan_Policy_2026.pdf p2 current]
    5. Fees and charges All fees b

MMR usually spreads results across more documents or sections. Neither is always better; check on your own questions.

**Score threshold:** an out-of-scope question should retrieve nothing rather than random chunks.

In [22]:
threshold_retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 4, "score_threshold": 0.35},
)

for q in ["What is the late fee on a credit card?", "Who won the cricket world cup?"]:
    found = threshold_retriever.invoke(q)
    print(f"{q} -> {len(found)} chunk(s) above threshold")

No relevant docs were retrieved using the relevance score threshold 0.35


What is the late fee on a credit card? -> 4 chunk(s) above threshold
Who won the cricket world cup? -> 0 chunk(s) above threshold


In [23]:
query = "Who won the cricket world cup?"

sim_retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
mmr_retriever = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 4, "fetch_k": 20, "lambda_mult": 1})

print("SIMILARITY:")
show(sim_retriever.invoke(query), width=80)
print("\nMMR:")
show(mmr_retriever.invoke(query), width=80)

SIMILARITY:
[Personal_Loan_Policy_2026.pdf p2 current]
    foreclosure charge. 6. Documents required Salaried applicants: PAN card, Aadhaar...
[Grievance_Redressal_Policy.pdf p1 current]
    Banking Ombudsman (RBI) If no reply within 30 days or the customer is not satisf...
[Grievance_Redressal_Policy.pdf p1 current]
    4. Compensation If a failed ATM or UPI transaction is not reversed within 5 days...
[Credit_Card_Terms_2026.pdf p1 current]
    Credit Card Terms and Conditions (2026) 1. Card variants and annual fees Card Jo...

MMR:
[Personal_Loan_Policy_2026.pdf p2 current]
    foreclosure charge. 6. Documents required Salaried applicants: PAN card, Aadhaar...
[Grievance_Redressal_Policy.pdf p1 current]
    Banking Ombudsman (RBI) If no reply within 30 days or the customer is not satisf...
[Grievance_Redressal_Policy.pdf p1 current]
    4. Compensation If a failed ATM or UPI transaction is not reversed within 5 days...
[Credit_Card_Terms_2026.pdf p1 current]
    Credit Card Terms an

The right threshold depends on the embedding model; 0.35 is a starting point for MiniLM with cosine. Too high and real questions get nothing.

**Metadata filter:** search only current documents.

In [28]:
current_retriever = vectorstore.as_retriever(search_kwargs={"k": 4, "filter": {"status": "current"}})
show(current_retriever.invoke("Can I close my personal loan early and what does it cost?"), width=80)

[Personal_Loan_Policy_2026.pdf p1 current]
    Personal Loan Policy (effective 1 April 2026) 1. Purpose and scope This policy d...
[Personal_Loan_Policy_2026.pdf p2 current]
    5. Fees and charges All fees below are exclusive of GST at 18%. Charge Amount Pr...
[Personal_Loan_Policy_2026.pdf p1 current]
    3. Loan amount and tenure The loan amount ranges from Rs 50,000 to Rs 25 lakh. T...
[Personal_Loan_Policy_2026.pdf p2 current]
    offer. Other applications are decided within 3 working days of receiving complet...


## Part 7: Generate grounded answers with citations

### Formatting the context

Each chunk is labelled with its source, page and status, so the LLM can cite and can tell current from superseded rules.

In [29]:
def format_docs(docs):
    blocks = []
    for d in docs:
        m = d.metadata
        blocks.append(f"[source: {m['source']}, page {m['page']}, status: {m['status']}]\n{d.page_content}")
    return "\n\n---\n\n".join(blocks)

print(format_docs(current_retriever.invoke("late payment fee on personal loan"))[:700])

[source: Personal_Loan_Policy_2026.pdf, page 2, status: current]
offer. Other applications are decided within 3 working days of receiving complete documents. If an
application is rejected, the customer is informed of the main reason in writing.
8. Missed payments
If an EMI is not paid on the due date, the late payment fee and penal charge apply, and the bank
reports the delay to credit bureaus. After 90 days of non-payment the account is classified as a
non-performing asset and recovery action may begin. Customers facing temporary financial difficulty
should contact the bank before the due date to discuss a revised repayment schedule.
9. Insurance
Loan protection insurance is optional. The b


### The grounded prompt

Every rule in the system message targets a specific failure:

| Rule | Prevents |
|---|---|
| answer only from the context | made-up facts from training data |
| cite source and page | unverifiable answers |
| prefer `current` over `superseded` | old rules being quoted |
| say you could not find it | guessing when retrieval missed |

In [30]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda

SYSTEM_PROMPT = (
    "You are a customer support assistant for Sahyadri Digital Bank.\n"
    "Rules:\n"
    "1. Answer ONLY from the context below. Do not use outside knowledge.\n"
    "2. Cite every fact as [file, page].\n"
    "3. If sources disagree, follow the one with status 'current'. Mention a superseded rule only if it still applies to the customer.\n"
    "4. If the context does not contain the answer, reply exactly: I could not find this in the bank's documents.\n"
    "5. Be concise: at most 5 sentences."
)

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
])

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

rag_chain = (
    RunnableParallel({"context": retriever | format_docs, "question": RunnablePassthrough()})
    | rag_prompt
    | llm
    | StrOutputParser()
)

print(rag_chain.invoke("What is the foreclosure charge on a personal loan?"))

The foreclosure charge is 2% of the outstanding principal [Personal_Loan_Policy_2026.pdf, page 2]. Foreclosure is not permitted before six EMIs have been paid [Personal_Loan_Policy_2026.pdf, page 2]. After six EMIs, the loan can be closed by paying the outstanding principal, interest up to the date of closure, and the foreclosure charge [Personal_Loan_Policy_2026.pdf, page 2].


In [31]:
print(rag_chain.invoke("who won the cricket world cup"))

I could not find this in the bank's documents.


### Returning the sources too

An application should show the user **which documents** the answer came from, not just the text. Keep the retrieved documents in the output with `.assign()`: it runs a step and **adds** its result to the dict instead of replacing it.

In [32]:
answer_step = (
    RunnableLambda(lambda x: {"context": format_docs(x["docs"]), "question": x["question"]})
    | rag_prompt
    | llm
    | StrOutputParser()
)

rag_with_sources = RunnableParallel({"docs": retriever, "question": RunnablePassthrough()}).assign(answer=answer_step)

result = rag_with_sources.invoke("I lost my credit card two days ago and someone used it. Do I have to pay?")

print("keys:", list(result.keys()))
print("\nANSWER:\n" + result["answer"])
print("\nSOURCES:")
for d in result["docs"]:
    print(f"  - {d.metadata['source']}, page {d.metadata['page']} ({d.metadata['status']})")

keys: ['docs', 'question', 'answer']

ANSWER:
You have zero liability for fraudulent transactions reported within 3 days of the card being lost [Credit_Card_Terms_2026.pdf, page 2]. Since you reported the loss within 3 days, you do not have to pay for fraudulent transactions [Credit_Card_Terms_2026.pdf, page 2].

SOURCES:
  - Credit_Card_Terms_2026.pdf, page 2 (current)
  - Credit_Card_Terms_2026.pdf, page 1 (current)
  - Credit_Card_Terms_2026.pdf, page 2 (current)
  - Credit_Card_Terms_2026.pdf, page 1 (current)


Streaming works on the whole chain, same as Session 2:

In [33]:
for piece in rag_chain.stream("How do I escalate a complaint that customer care did not resolve?"):
    print(piece, end="", flush=True)
print()

If your complaint is not resolved by customer care, you can escalate it to the Grievance Officer. If it's still not resolved or you're not satisfied, you can escalate it to the Principal Nodal Officer. The Grievance Officer will respond within 15 days, and the Principal Nodal Officer will provide a final bank response within 30 days of the original complaint [source: Grievance_Redressal_Policy.pdf, page 1]. If you do not receive a reply within 30 days or are not satisfied with the final response, you may approach the RBI Ombudsman [source: Grievance_Redressal_Policy.pdf, page 1].


## Part 8: Failure modes, and how to debug them

When a RAG answer is wrong, ask in this order:

1. **Was the right chunk retrieved?** (retrieval problem)
2. **Was the chunk complete and readable?** (loading or chunking problem)
3. **Did the LLM use it correctly?** (prompt or model problem)

A debug helper that answers question 1 and 3 side by side:

In [34]:
def debug(question, retr=retriever):
    docs = retr.invoke(question)
    print("QUESTION:", question)
    print("\nRETRIEVED:")
    show(docs, width=90)
    answer = (rag_prompt | llm | StrOutputParser()).invoke({"context": format_docs(docs), "question": question})
    print("\nANSWER:\n" + answer)

### 8.1 Conflicting documents

The 2024 and 2026 policies disagree. Two fixes, best used together:
- **Prompt rule** (already in the system prompt): prefer `current`.
- **Metadata filter**: do not retrieve superseded documents at all, unless the question is about an old loan.

In [35]:
q = "What is the foreclosure charge on a personal loan?"
print("=== all documents ===")
debug(q)
print("\n=== current documents only ===")
debug(q, retr=current_retriever)

=== all documents ===
QUESTION: What is the foreclosure charge on a personal loan?

RETRIEVED:
[Personal_Loan_Policy_2026.pdf p2 current]
    5. Fees and charges All fees below are exclusive of GST at 18%. Charge Amount Processing f...
[Personal_Loan_Policy_2024.pdf p1 superseded]
    Personal Loan Policy 2024 (superseded) Note: this policy was replaced by the Personal Loan...
[Personal_Loan_Policy_2026.pdf p2 current]
    foreclosure charge. 6. Documents required Salaried applicants: PAN card, Aadhaar, salary s...
[Personal_Loan_Policy_2026.pdf p2 current]
    the customer chooses it, the premium is shown separately and can be paid upfront or added ...

ANSWER:
The foreclosure charge is 2% of the outstanding principal [Personal_Loan_Policy_2026.pdf, page 2]. Foreclosure is not permitted before six EMIs have been paid [Personal_Loan_Policy_2026.pdf, page 2]. After six EMIs, the loan can be closed by paying the outstanding principal, interest up to the date of closure, and the foreclosu

### 8.2 Out of scope

The documents say nothing about this. A grounded chain should refuse rather than invent.

In [36]:
debug("What is the current RBI repo rate?")

QUESTION: What is the current RBI repo rate?

RETRIEVED:
[Personal_Loan_Policy_2026.pdf p1 current]
    3. Loan amount and tenure The loan amount ranges from Rs 50,000 to Rs 25 lakh. The maximum...
[Personal_Loan_Policy_2024.pdf p1 superseded]
    Personal Loan Policy 2024 (superseded) Note: this policy was replaced by the Personal Loan...
[Credit_Card_Terms_2026.pdf p1 current]
    an interest-free period of 20 to 50 days depending on the purchase date. 3. Interest and f...
[Grievance_Redressal_Policy.pdf p1 current]
    Banking Ombudsman (RBI) If no reply within 30 days or the customer is not satisfied with t...

ANSWER:
I could not find this in the bank's documents.


### 8.3 Hallucination without grounding rules

Our documents cover **personal** loans only. Compare a naive prompt with the grounded one on a home loan question.

In [37]:
naive_prompt = ChatPromptTemplate.from_messages([
    ("human", "Use this context to help answer.\n\n{context}\n\nQuestion: {question}"),
])
naive_chain = (
    RunnableParallel({"context": retriever | format_docs, "question": RunnablePassthrough()})
    | naive_prompt | llm | StrOutputParser()
)

q = "What is the foreclosure charge on a home loan?"
print("NAIVE PROMPT:\n", naive_chain.invoke(q))
print("\nGROUNDED PROMPT:\n", rag_chain.invoke(q))

NAIVE PROMPT:
 The provided documents discuss the foreclosure charges for a **personal loan**, not a home loan.

Based on the **Personal Loan Policy 2026**, the foreclosure charge is **2% of the outstanding principal**. This is not allowed before 6 EMIs have been paid.

GROUNDED PROMPT:
 I could not find this in the bank's documents.


The naive version often answers with the personal loan figure or a general fact about home loans. The grounded version should say it could not find it. Results vary by model; run it a few times.

### 8.4 Chunking that splits a fact

With tiny chunks and no overlap, a table label and its value can land in different chunks. Then no single chunk answers the question.

In [38]:
bad_splitter = RecursiveCharacterTextSplitter(chunk_size=60, chunk_overlap=0)
fees_page = next(p for p in clean_pages
                 if p.metadata["source"] == "Personal_Loan_Policy_2026.pdf" and "Foreclosure (full prepayment)" in p.page_content)
bad_chunks = bad_splitter.split_text(fees_page.page_content)

for i, c in enumerate(bad_chunks):
    if "Foreclosure" in c or "outstanding principal" in c:
        print(f"chunk {i}: {c!r}")

chunk 4: 'Foreclosure (full prepayment)'
chunk 5: '2% of the outstanding principal; not allowed before 6 EMIs'
chunk 11: 'Foreclosure is not allowed in the first six months of the'
chunk 13: 'the loan at any time by paying the outstanding principal,'


"Foreclosure (full prepayment)" and "2% of the outstanding principal" are in different chunks. A search for "foreclosure charge" finds the label chunk, which has no number in it. Sensible chunk size and overlap prevent this.

### 8.5 Tables and numbers

Questions that need a lookup in a flattened table (credit score band -> rate) are harder for both retrieval and the LLM. Check the answer against the PDF.

In [39]:
debug("What interest rate will I get on a personal loan with a credit score of 750?", retr=current_retriever)

QUESTION: What interest rate will I get on a personal loan with a credit score of 750?

RETRIEVED:
[Personal_Loan_Policy_2026.pdf p1 current]
    3. Loan amount and tenure The loan amount ranges from Rs 50,000 to Rs 25 lakh. The maximum...
[Personal_Loan_Policy_2026.pdf p1 current]
    2. Eligibility Salaried applicants must be between 21 and 60 years of age at the time of a...
[Personal_Loan_Policy_2026.pdf p1 current]
    Personal Loan Policy (effective 1 April 2026) 1. Purpose and scope This policy describes h...
[Personal_Loan_Policy_2026.pdf p2 current]
    5. Fees and charges All fees below are exclusive of GST at 18%. Charge Amount Processing f...

ANSWER:
With a credit score of 750, the annual interest rate on a personal loan is 11.75% [source: Personal_Loan_Policy_2026.pdf, page 1]. If you are an existing customer with a salary account at Sahyadri Digital Bank for at least 12 months, you may receive a discount of 0.25% on this rate [source: Personal_Loan_Policy_2026.pdf, page 

Correct answer: score 750 is in the "740 to 779" band, so **11.75%** (or 11.5% with the salary-account discount). If the model picks the wrong band, the fix is usually better table extraction (keep rows together, or convert tables to "label: value" sentences while cleaning) rather than a bigger model.

## Part 9: Evaluate with a small test set

"It looks right on three questions" is not evaluation. Write 8 to 20 questions **with known answers** from the documents, and measure two things separately:

- **Retrieval hit**: was the expected page among the retrieved chunks?
- **Answer check**: does the answer contain the key fact?

Rerun this every time you change chunk size, k, the embedding model or the prompt. Now changes are measured, not guessed.

In [40]:
eval_set = [
    {"q": "What is the foreclosure charge on a personal loan?", "source": "Personal_Loan_Policy_2026.pdf", "page": 2, "expect": "2%"},
    {"q": "Is part-prepayment of a personal loan free?", "source": "Personal_Loan_Policy_2026.pdf", "page": 2, "expect": "6 EMI"},
    {"q": "What is the minimum credit score for a personal loan?", "source": "Personal_Loan_Policy_2026.pdf", "page": 1, "expect": "700"},
    {"q": "What is the processing fee on a personal loan?", "source": "Personal_Loan_Policy_2026.pdf", "page": 2, "expect": "1.5%"},
    {"q": "When is the credit card payment due?", "source": "Credit_Card_Terms_2026.pdf", "page": 1, "expect": "25th"},
    {"q": "What interest is charged if I pay only part of my card bill?", "source": "Credit_Card_Terms_2026.pdf", "page": 1, "expect": "3.5%"},
    {"q": "How much is the cash advance fee on the credit card?", "source": "Credit_Card_Terms_2026.pdf", "page": 1, "expect": "2.5%"},
    {"q": "What is my liability if my stolen card is misused and I report it in 2 days?", "source": "Credit_Card_Terms_2026.pdf", "page": 2, "expect": "zero"},
    {"q": "How long does the Grievance Officer take to respond?", "source": "Grievance_Redressal_Policy.pdf", "page": 1, "expect": "15 days"},
    {"q": "What compensation do I get if a failed UPI transaction is not reversed?", "source": "Grievance_Redressal_Policy.pdf", "page": 1, "expect": "100"},
]

def evaluate(retr, chain_prompt=rag_prompt):
    rows = []
    for item in eval_set:
        docs = retr.invoke(item["q"])
        hit = any(d.metadata["source"] == item["source"] and d.metadata["page"] == item["page"] for d in docs)
        answer = (chain_prompt | llm | StrOutputParser()).invoke({"context": format_docs(docs), "question": item["q"]})
        rows.append({
            "question": item["q"][:55],
            "retrieval_hit": hit,
            "answer_ok": item["expect"].lower() in answer.lower(),
            "answer": answer[:80].replace("\n", " "),
        })
    return pd.DataFrame(rows)

In [41]:
report = evaluate(current_retriever)
print(f"retrieval hit rate: {report['retrieval_hit'].mean():.0%}")
print(f"answer check rate : {report['answer_ok'].mean():.0%}")
report

retrieval hit rate: 100%
answer check rate : 100%


,question,retrieval_hit,answer_ok,answer
0,What is the foreclosure charge on a personal l...,True,True,The foreclosure charge on a personal loan is 2...
1,Is part-prepayment of a personal loan free?,True,True,"Yes, part-prepayment of a personal loan is fre..."
2,What is the minimum credit score for a persona...,True,True,The minimum credit score required for a person...
3,What is the processing fee on a personal loan?,True,True,The processing fee for a personal loan is 1.5%...
4,When is the credit card payment due?,True,True,The credit card payment is due on the 25th of ...
5,What interest is charged if I pay only part of...,True,True,"If only part of the amount due is paid, intere..."
6,How much is the cash advance fee on the credit...,True,True,"The cash advance fee is 2.5% of the amount, wi..."
7,What is my liability if my stolen card is misu...,True,True,If your stolen card is misused and you report ...
8,How long does the Grievance Officer take to re...,True,True,The Grievance Officer will respond within 15 d...
9,What compensation do I get if a failed UPI tra...,True,True,If a failed UPI transaction is not reversed wi...


Read the failures row by row:
- `retrieval_hit = False`: fix retrieval (chunking, k, embedding model, cleaning).
- `retrieval_hit = True` but `answer_ok = False`: fix the prompt, or check whether the answer is right in different words (string matching is crude; Exercise 2 uses an LLM as judge).

This is a first evaluation, not a complete one. Production teams use larger test sets and tools such as RAGAS or LangSmith, but the idea is the same.

## Part 10: Capstone part 1 - RAG over your own documents

Put 1 to 5 PDF, TXT or MD files from your domain into a folder called `my_docs/` next to this notebook (public or dummy data only), then run the cells below.

The function bundles today's pipeline. Everything you learned is inside it: loading, cleaning, metadata, chunking, deterministic ids, storing, and a grounded chain that returns sources.

In [ ]:
def load_folder(folder):
    docs = []
    for path in sorted(glob.glob(os.path.join(folder, "*"))):
        ext = os.path.splitext(path)[1].lower()
        if ext == ".pdf":
            loaded = load_pdf(path)
        elif ext in (".txt", ".md"):
            with open(path, encoding="utf-8") as f:
                loaded = [Document(page_content=f.read(), metadata={"source": path, "page": 0})]
        else:
            print("skipped (unsupported type):", path)
            continue
        for d in loaded:
            page = int(d.metadata.get("page", 0)) + 1
            docs.append(Document(page_content=d.page_content.strip(),
                                 metadata={"source": os.path.basename(path), "page": page, "status": "current"}))
    return docs


def build_rag(folder, collection_name, domain_description, k=4, chunk_size=800, chunk_overlap=120):
    docs = [d for d in load_folder(folder) if d.page_content]
    pieces = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap,
                                            add_start_index=True).split_documents(docs)

    if collection_name in [c.name for c in client.list_collections()]:
        client.delete_collection(collection_name)
    store = Chroma(client=client, collection_name=collection_name, embedding_function=embeddings,
                   collection_configuration={"hnsw": {"space": "cosine"}})
    store.add_documents(pieces, ids=[f"{p.metadata['source']}-p{p.metadata['page']}-{p.metadata['start_index']}" for p in pieces])
    print(f"indexed {len(docs)} pages / sections into {len(pieces)} chunks")

    prompt = ChatPromptTemplate.from_messages([
        ("system", f"You are an assistant for {domain_description}. " + SYSTEM_PROMPT.split("\n", 1)[1]),
        ("human", "Context:\n{context}\n\nQuestion: {question}"),
    ])
    retr = store.as_retriever(search_kwargs={"k": k})
    answer = (RunnableLambda(lambda x: {"context": format_docs(x["docs"]), "question": x["question"]})
              | prompt | llm | StrOutputParser())
    return RunnableParallel({"docs": retr, "question": RunnablePassthrough()}).assign(answer=answer)

In [ ]:
MY_DOCS = "my_docs"

if os.path.isdir(MY_DOCS) and os.listdir(MY_DOCS):
    my_rag = build_rag(MY_DOCS, "capstone_docs", domain_description="my domain documents")
    out = my_rag.invoke("Write a question about your documents here")
    print(out["answer"])
    print("sources:", sorted({(d.metadata["source"], d.metadata["page"]) for d in out["docs"]}))
else:
    print(f"Create a folder '{MY_DOCS}' next to this notebook and put your documents in it.")
    print("Meanwhile, the same function works on today's documents:")
    demo_rag = build_rag("docs", "demo_docs", domain_description="Sahyadri Digital Bank customers")
    print(demo_rag.invoke("How long does customer care take to respond to a complaint?")["answer"])

**Capstone checklist for this week**
- [ ] 1 to 5 documents in `my_docs/`
- [ ] Look at the extracted text: headers, footers, tables? Add a cleaner if needed.
- [ ] Write 5 test questions with known answers and run them through `evaluate`-style checks.
- [ ] Note one failure and what you think causes it. Bring it to the next session.

## Part 11: Exercises

1. **Easy** - Rerun the evaluation with `k=2` and `k=8`. What happens to retrieval hit rate and to answer quality? Why might more context make answers *worse*?
2. **Medium** - LLM as judge: create a Pydantic model `Judgement(correct: bool, grounded: bool, reason: str)` and use `with_structured_output` to grade each answer against the expected fact and the retrieved context. Compare with the string check.
3. **Challenge** - Improve table handling: in `clean_text`, convert the fee table lines into "label: value" sentences for the loan policy, re-index, and rerun the evaluation. Did the credit-score question (8.5) improve?